<a href="https://colab.research.google.com/github/slavyolov/Algorithms/blob/main/PuLP_Basic_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Basic Example

In [ ]:
import pulp as pl

# Create model
model = pl.LpProblem("Production_Planning", pl.LpMaximize)

# Products
products = ["Product_A", "Product_B", "Product_C"]
profit = {"Product_A": 10, "Product_B": 12, "Product_C": 14}

# Resource limits
resources = ["Labor", "Material_X", "Material_Y"]
available = {"Labor": 480, "Material_X": 320, "Material_Y": 250}

# Resource usage per unit
usage = {
    ("Product_A", "Labor"): 5,
    ("Product_B", "Labor"): 4,
    ("Product_C", "Labor"): 6,
    ("Product_A", "Material_X"): 3,
    ("Product_B", "Material_X"): 5,
    ("Product_C", "Material_X"): 2,
    ("Product_A", "Material_Y"): 2,
    ("Product_B", "Material_Y"): 1,
    ("Product_C", "Material_Y"): 4
}

# Decision variables
production = {
    p: pl.LpVariable(f"Produce_{p}", lowBound=0, cat="Integer")
    for p in products
}

# Objective: maximize profit
model += pl.lpSum(profit[p] * production[p] for p in products)

# Constraints: resource capacity
for r in resources:
    model += pl.lpSum(usage[(p, r)] * production[p] for p in products) <= available[r]

# Model Info
print("--- Model Statistics ---")
print(f"Model Name: {model.name}")
print(f"Sense: {'Maximize' if model.sense == pl.LpMaximize else 'Minimize'}")
print(f"Variables: {len(model.variables())}")
print(f"Constraints: {len(model.constraints)}")

# Detailed inspection of constraints
print("\n--- Constraints Detail ---")
for name, constraint in model.constraints.items():
    print(f"{name}: {constraint}")

## Solve the model

In [ ]:
# Solve
model.solve()

# Results
print("\n--- Status: ", pl.LpStatus[model.status])
print("\n--- Products Manufactured: ")
for p in products:
    print(p, production[p].value())
print("\n--- Total profit ---\n", pl.value(model.objective))

# GUI, Solver

In [ ]:
import pulp as pl
import ipywidgets as widgets
from IPython.display import display, clear_output

# Initial values for profit and resources
initial_profit = {"Product_A": 10, "Product_B": 12, "Product_C": 14}
initial_available = {"Labor": 480, "Material_X": 320, "Material_Y": 250}

profit_widgets = {
    "Product_A": widgets.IntSlider(value=initial_profit["Product_A"], min=0, max=50, step=1, description="Profit A"),
    "Product_B": widgets.IntSlider(value=initial_profit["Product_B"], min=0, max=50, step=1, description="Profit B"),
    "Product_C": widgets.IntSlider(value=initial_profit["Product_C"], min=0, max=50, step=1, description="Profit C"),
}

resource_widgets = {
    "Labor": widgets.IntSlider(value=initial_available["Labor"], min=0, max=1000, step=10, description="Labor"),
    "Material_X": widgets.IntSlider(value=initial_available["Material_X"], min=0, max=1000, step=10, description="Mat X"),
    "Material_Y": widgets.IntSlider(value=initial_available["Material_Y"], min=0, max=1000, step=10, description="Mat Y"),
}

solve_button = widgets.Button(description="Solve Model", button_style="success")
reset_button = widgets.Button(description="Reset Values", button_style="info")
output = widgets.Output()

def solve_model(_):
    with output:
        clear_output()

        profit = {p: profit_widgets[p].value for p in products}
        available = {r: resource_widgets[r].value for r in resources}

        model = pl.LpProblem("Production_Planning", pl.LpMaximize)

        production = {
            p: pl.LpVariable(f"Produce_{p}", lowBound=0, cat="Integer")
            for p in products
        }

        model += pl.lpSum(profit[p] * production[p] for p in products)

        for r in resources:
            model += pl.lpSum(usage[(p, r)] * production[p] for p in products) <= available[r]

        model.solve()

        print("PuLP Version:", pl.__version__)
        print("Status:", pl.LpStatus[model.status])
        for p in products:
            print(f"{p}: {production[p].value()}")
        print("Total profit:", pl.value(model.objective))

def reset_values(_):
    for p in products:
        profit_widgets[p].value = initial_profit[p]
    for r in resources:
        resource_widgets[r].value = initial_available[r]
    with output:
        clear_output()
        print("Values reset to defaults.")

solve_button.on_click(solve_model)
reset_button.on_click(reset_values)

ui = widgets.VBox([
    widgets.HTML("<h3>Production Planning Optimizer</h3>"),
    widgets.HTML("<b>Profit per product</b>"),
    *profit_widgets.values(),
    widgets.HTML("<b>Available resources</b>"),
    *resource_widgets.values(),
    widgets.HBox([solve_button, reset_button]),
    output
])

display(ui)